# 01 — Indexing and Lookup

**Purpose:** Demonstrate every way a user can index into a `Trace` to retrieve records,
plus the structured accessor views for layers, ops, modules, params, and buffers.

**Surfaces covered:**
- [ ] `trace[int]` — ordinal indexing (positive and negative)
- [ ] `trace[label]` — exact string label lookup
- [ ] `trace[substring]` — unambiguous substring lookup
- [ ] `trace[module_path]` — module-path lookup (e.g. `"fc"`)
- [ ] Ambiguous lookup raises `ValueError` (inspect message)
- [ ] `op.raw_label` and by-raw-index lookup
- [ ] `op.lookup_keys` — the full set of valid keys for an op
- [ ] `trace.layer_labels` / `trace.op_labels` accessors
- [ ] `trace.layers` — `LayerAccessor` repr
- [ ] `trace.ops` — `TraceOpAccessor` repr
- [ ] `trace.modules` — `ModuleAccessor` repr
- [ ] `trace.params` — param dict repr
- [ ] `trace.buffers` — buffer dict repr
- [ ] `Layer.__repr__` — multi-pass layer record
- [ ] `Op.__repr__` — single-pass op record

## 1. Environment setup

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. Capture a simple trace for indexing demos

Using `tiny_mlp` (8->relu->4) — small enough to print every label, simple enough to
reason about. We capture once and reuse for all indexing examples.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

print(repr(trace))
print()
print("layer_labels :", trace.layer_labels)
print("op_labels    :", trace.op_labels)

## 3. Integer indexing — `trace[int]`

Positive indices count from input; negative indices count back from the output.
`trace[0]` is the input op; `trace[-1]` is the output op.

In [ ]:
# Positive ordinal
op0 = trace[0]
print("=== trace[0] ===")
print(repr(op0))
print()

# Negative ordinal (last = output)
op_last = trace[-1]
print("=== trace[-1] ===")
print(repr(op_last))
print()

# Middle element by index
op_mid = trace[2]
print("=== trace[2] ===")
print(repr(op_mid))

## 4. Exact label lookup — `trace[label]`

Pass a string from `trace.layer_labels` for an exact match.  
The returned object is the `Layer` record for that label (which groups all passes
of the same logical op across loop iterations).

In [ ]:
relu_label = trace.layer_labels[2]  # 'relu_1_2'
layer = trace[relu_label]

print(f"trace['{relu_label}'] -> type: {type(layer).__name__}")
print()
print(repr(layer))

## 5. Op-label lookup — `trace[op_label]` (pass-qualified)

`op_labels` include the pass suffix (e.g. `relu_1_2:1`).  Using an op-label returns
an `Op` record (single execution) rather than a `Layer` (all-passes group).

In [ ]:
op_lbl = trace.op_labels[2]  # 'relu_1_2:1'
op = trace[op_lbl]

print(f"trace['{op_lbl}'] -> type: {type(op).__name__}")
print()
print(repr(op))

## 6. Substring lookup

If the substring uniquely identifies one layer, TorchLens resolves it automatically.
The example here uses `"relu"` which matches exactly one layer in `tiny_mlp`.

In [ ]:
found = trace["relu"]
print(f"trace['relu'] -> type: {type(found).__name__}")
print(f"layer_label  : {found.layer_label}")
print(f"out.shape    : {found.out.shape}")

## 7. Ambiguous substring lookup raises `ValueError`

When a substring matches multiple layers, TorchLens raises a `ValueError`
(formerly named `AmbiguousOpLookupError`; now it IS `ValueError`).  
The error message lists all matching layer labels so the user can refine their key.

In [ ]:
import torch.nn as nn


class TwoReluModel(nn.Module):
    """Two independent relu ops => ambiguous substring 'relu'."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.relu(x) + torch.relu(x)


m2, x2 = TwoReluModel(), torch.randn(4)
trace2 = tl.trace(m2, x2)

print("layer_labels:", trace2.layer_labels)
print()

try:
    trace2["relu"]
except ValueError as e:
    print("ValueError caught (ambiguous lookup):")
    print(" ", e)

# NOTE: the spec mentions AmbiguousOpLookupError but the runtime raises ValueError
# (see GAPs section below for details)

## 8. Module-path lookup

Using a module address (e.g. `"in_proj"`) as the lookup key finds the layer that
is the output of that module call.

In [ ]:
module_found = trace["in_proj"]
print(f"trace['in_proj'] -> type: {type(module_found).__name__}")
print(f"layer_label    : {module_found.layer_label}")
print(f"out.shape      : {module_found.out.shape}")

## 9. `op.raw_label` and by-raw-index lookup

`raw_label` is the pre-deduplication label assigned during tracing (e.g. `linear_1_2`).
It can differ from `layer_label` when two ops share the same functional label but are
executed at different trace steps.  The raw label is appended with `_raw` to form a
valid raw lookup key.

In [ ]:
op = trace[1]  # linear_1_1 op
print(f"op.layer_label : {op.layer_label}")
print(f"op.raw_label   : {op.raw_label}")
print(f"op.raw_index   : {op.raw_index}")
print()

# Lookup by raw label (append _raw suffix)
raw_key = op.raw_label + "_raw"
try:
    found_raw = trace[raw_key]
    print(f"trace['{raw_key}'] -> layer_label: {found_raw.layer_label}")
except Exception as e:
    print(f"trace['{raw_key}'] raised: {type(e).__name__}: {e}")

print()
print("Full lookup_keys for this op:")
print(" ", op.lookup_keys)

## 10. `trace.layers` — the Layer accessor

`trace.layers` is a `LayerAccessor` that shows every layer with its function type,
output shape, and op count.  Indexing into it also works.

In [ ]:
print(repr(trace.layers))
print()

# Access via accessor
layer_via_acc = trace.layers["linear_1_1"]
print("trace.layers['linear_1_1'] type:", type(layer_via_acc).__name__)
print(repr(layer_via_acc))

## 11. `trace.ops` — the Op accessor

In [ ]:
print(repr(trace.ops))
print()

# Access one op by pass-qualified label
op_via_acc = trace.ops["linear_1_1:1"]
print("trace.ops['linear_1_1:1'] type:", type(op_via_acc).__name__)
print("layer_label:", op_via_acc.layer_label)
print("out.shape  :", op_via_acc.out.shape)

## 12. `trace.modules` — the Module accessor

In [ ]:
print(repr(trace.modules))
print()

mod = trace.modules["in_proj"]
print("Module repr:")
print(repr(mod))

## 13. `trace.params` — parameter records

In [ ]:
print(repr(trace.params))
print()

# Show one Param repr
param = trace.params["in_proj.weight"]
print("Param repr:")
print(repr(param))

## 14. `trace.buffers` — buffer records

`tiny_mlp` has no buffers.  Use `batch_norm` which has `running_mean`, `running_var`,
and `num_batches_tracked`.

In [ ]:
model_bn, x_bn = ZOO["batch_norm"]()
trace_bn = tl.trace(model_bn, x_bn)

print("Buffer names:", list(trace_bn.buffers.keys()))
print()
print(repr(trace_bn.buffers))
print()

# Show one Buffer repr
if trace_bn.buffers:
    buf_key = list(trace_bn.buffers.keys())[0]
    buf = trace_bn.buffers[buf_key]
    print(f"Buffer repr for '{buf_key}':")
    print(repr(buf))

## 15. `Layer.__repr__` vs `Op.__repr__` — structural comparison

Use `demo_model` which has a loop submodule (same op executed 3x). The `Layer` record
groups all three passes; each `Op` record is one pass.  This shows the difference
between `trace[label]` (returns `Layer`) and `trace[op_label]` (returns `Op`).

In [ ]:
model_d, x_d = ZOO["demo_model"]()
trace_d = tl.trace(model_d, x_d)

# Find a layer with multiple passes (loop)
multi_pass_labels = [lbl for lbl in trace_d.layer_labels if trace_d[lbl].num_passes > 1]
print("Layers with >1 pass:", multi_pass_labels)

if multi_pass_labels:
    mp_lbl = multi_pass_labels[0]
    layer_mp = trace_d[mp_lbl]  # Layer record (all passes)
    print()
    print(f"=== Layer.__repr__ for '{mp_lbl}' ({layer_mp.num_passes} passes) ===")
    print(repr(layer_mp))
    print()

    # First Op record
    op_mp = trace_d[mp_lbl + ":1"]  # first pass
    print(f"=== Op.__repr__ for '{mp_lbl}:1' ===")
    print(repr(op_mp))

---

## ⚠️ GAPs / ergonomic smells

- **`AmbiguousOpLookupError` is not importable from `torchlens`.**  
  The spec names it as a distinct class (`AmbiguousOpLookupError`) and says it is raised
  on ambiguous substring lookups.  In practice, `tl.trace(...)[ambiguous_key]` raises a
  plain `ValueError` (not a subclass with a distinct name).  `dir(tl)` shows only
  `ActivationLookup`, not `AmbiguousOpLookupError`.  This makes it impossible for callers
  to write `except AmbiguousOpLookupError` for targeted handling.

- **`trace[int]` returns an `Op`, but `trace[layer_label]` returns a `Layer`.** The repr
  header for the `Op` from integer indexing reads `"Layer X, operation N/M"` which implies
  it is a `Layer`, yet `type(trace[0]).__name__` is `'Op'`.  This wording is confusing:
  "Layer input_1, operation 0/3" sounds like it returned the Layer but the object is an Op.

- **`trace.buffers` is empty for `tiny_mlp`** (no buffers — expected).  The repr shows
  `{}` which is correct but new users may think buffers aren't tracked at all.  A brief
  message like `BufferAccessor(0 buffers)` (analogous to `ModuleAccessor(N modules)`) would
  make it clearer that the accessor exists but there is nothing to show.

- **Module-path lookup resolves to the layer that is the output of that module call**,
  which is ergonomically reasonable, but is undocumented in the repr/error messages.
  A first-time user who writes `trace['fc']` expecting a `Module` record
  (and gets a `Layer`) will be surprised.  Consider a pointer in the accessor repr or
  the error message for non-matching module paths.